In [3]:
import os
import math
import torch
from torch.nn.functional import softplus
from datetime import datetime
from tensorboardX import SummaryWriter
from nn.neural_network_dens2 import NeuralNetwork
from training.parse_command_line_arguments import parse_command_line_arguments
from training.util import generate_id, empty_error_dict, compute_error_dict
from training.density_dataset import AtomsDensityData
from training.hamiltonian_dataset import seeded_random_split
from training.exponential_moving_average import ExponentialMovingAverage
from training.lookahead import Lookahead
from training.batch_loader import BatchLoader
from nn.modules.spherical_harmonics_expansion import SphericalHarmonicsExpansion
import numpy as np
import time
from functools import partial
from training.grids import cubical_grid, cubical_sampling, spherical_grid

%load_ext autoreload
%autoreload 2

In [4]:
directory = '2020-08-20_LuCuJBCl'  # load directory name
model_name = 'LuCuJBCl'

#directory = '2020-07-30_ihEqX0KV'  # load directory name
#model_name = 'ihEqX0KV'

#directory = '2020-04-30_To0wdEze'
checkpoint_dir = os.path.join(
    directory, 'checkpoints')  # checkpoint directory
# load latest checkpoint
checkpoint = torch.load(os.path.join(
    checkpoint_dir, 'latest_checkpoint.pth'), map_location='cpu')
latest_checkpoint = checkpoint['epoch']
ID = checkpoint['ID']  # load ID
args = checkpoint['args']  # overwrite args
args.use_gpu = False
args.load_from = os.path.join(directory, 'best_' + model_name + '.pth')

In [5]:
use_gpu = args.use_gpu and torch.cuda.is_available()

# load dataset(s)
print("loading density from" + args.dens_dataset + "...")
print("loading atoms from" + args.np_dataset + "...")

# density_file = '/home/mihail/data/water_rot/full_densities.hdf5'
# np_file = 'h2o_overlap_static.npy'

args.num_workers = 0
sph_grid_fn = partial(spherical_grid, level=5)

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=args.density_subsamples,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=sph_grid_fn)



loading density fromdatasets/h2o_dynamic_pyscf_dft.npy...
loading atoms fromdatasets/h2o_dynamic_centered.npy...
Starting atomsdata density init
Some variables
level 5
finished init


In [6]:
cube_grid_fn = partial(cubical_grid, nx=50, ny=50, nz=50,
                       extent=np.array([4.1483, 4.1483, 4.1483]),
                       origin=np.array([-2.0318, -2.0318 , -2.0318]))
cube_sampling_fn = cubical_sampling

valid_cube_dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                                      orbitals_path=args.orbitals_file,
                                      density_n_samp=10000000000,
                                      required_properties=['density'],
                                      center_positions=False,
                                      radial_coeffs_file=args.radial_coeffs_file,
                                      dtype=args.dtype,
                                      grid_fn=cube_grid_fn,
                                      sampling_fn=cube_sampling_fn)


Starting atomsdata density init
Some variables
finished init


In [7]:
# determine weights of different quantities for scaling loss
sampler = torch.utils.data.BatchSampler(torch.utils.data.SequentialSampler(dataset),
                                              batch_size=6, drop_last=False)

data_loader = BatchLoader(dataset, batch_sampler=sampler,
                                num_workers=args.num_workers, pin_memory=use_gpu)
loss_weights = {}
loss_weights['density'] = args.density_weight
loss_weights['energy'] = args.energy_weight

valid_cube_sampler = torch.utils.data.BatchSampler(torch.utils.data.RandomSampler(valid_cube_dataset),
                                                   batch_size=args.valid_batch_size, drop_last=False)
valid_cube_loader = BatchLoader(valid_cube_dataset, batch_sampler=valid_cube_sampler,
                                num_workers=args.num_workers, pin_memory=use_gpu)


In [8]:
rad_iterator = iter(data_loader)
cube_iterator = iter(valid_cube_loader)

In [9]:
# define model
if args.load_from is None:
    equiv_model = NeuralNetwork(
        orbitals=dataset.orbitals,
        order=args.order,
        num_features=args.num_features,
        num_basis_functions=args.num_basis_functions,
        num_modules=args.num_modules,
        num_residual_pre_x=args.num_residual_pre_x,
        num_residual_post_x=args.num_residual_post_x,
        num_residual_pre_vi=args.num_residual_pre_vi,
        num_residual_pre_vj=args.num_residual_pre_vj,
        num_residual_post_v=args.num_residual_post_v,
        num_residual_output=args.num_residual_output,
        num_radial_components=args.num_radial_components,
        basis_functions=args.basis_functions,
        cutoff=args.cutoff,
        activation=args.activation)
    expansion_model = SphericalHarmonicsExpansion(dataset.orbitals,
                                                  radial_coeffs=dataset.radial_coeffs)
else:
    equiv_model = NeuralNetwork(load_from=args.load_from)
    expansion_model = SphericalHarmonicsExpansion(dataset.orbitals, radial_coeffs=dataset.radial_coeffs, constraint_type=args.expansion_constraint)

    

# convert the model to the correct dtype
equiv_model.to(args.dtype)
expansion_model.to(args.dtype)

# send model to GPU (if use_gpu is True)
if use_gpu:
    equiv_model.cuda()
    expansion_model.cuda()


saved state dict_keys(['state_dict', 'orbitals', 'order', 'num_features', 'num_basis_functions', 'num_radial_components', 'num_modules', 'num_residual_pre_x', 'num_residual_post_x', 'num_residual_pre_vi', 'num_residual_pre_vj', 'num_residual_post_v', 'num_residual_output', 'basis_functions', 'cutoff', 'activation', 'Zmax'])
An orbital with L=5 was found, but the neural network was initialized with L=6
The neural network SHOULD have at least twice the order of the maximum order orbital for good results!
orbital_spec [[(8, 11, 0), (8, 8, 1), (8, 6, 2), (8, 4, 3), (8, 3, 4), (8, 2, 5)], [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)], [(1, 5, 0), (1, 4, 1), (1, 4, 2), (1, 3, 3), (1, 2, 4)]]
cg_matrix shape torch.Size([121, 121, 121])
L_counts [16, 12, 10, 7, 5, 2, 0, 0, 0, 0, 0, 0, 0]
L_dict {(8, 0): range(0, 11), (8, 1): range(0, 8), (8, 2): range(0, 6), (8, 3): range(0, 4), (8, 4): range(0, 3), (8, 5): range(0, 2), (1, 0): range(11, 16), (1, 1): range(8, 12), (1, 2): range(6, 10

In [10]:
equiv_module = equiv_model
expansion_module = expansion_model

# for keeping an exponential moving average of the model parameters (usually leads to better models)
if checkpoint is not None:  # no checkpoint is specified
    step = checkpoint['step']
    epoch = checkpoint['epoch']
    best_errors = checkpoint['best_errors']
    valid_errors = checkpoint['valid_errors']
    equiv_module.load_state_dict(checkpoint['model_state_dict'])
# or initialize step / epoch to 0 and errors to infinity
else:
    step = 0
    epoch = 0
    best_errors = empty_error_dict(loss_weights, fill_value=math.inf)
    valid_errors = empty_error_dict(loss_weights, fill_value=math.inf)



In [75]:
import torch
import torch.nn as nn
from nn.spherical_harmonics import spherical_harmonics


class TestSphericalHarmonicsExpansion(nn.Module):

    def __init__(self, orbitals, radial_coeffs=None, constraint_type=None):
        super().__init__()
        self.orbitals = orbitals
        self.order_max = 0
        self.constraint_type = constraint_type
        print('constraint type',  self.constraint_type)
        self.n_electrons = 0
        for i in range(len(self.orbitals)):
            self.n_electrons += self.orbitals[i][0][0]
            for z, _, l in self.orbitals[i]:
                if l > self.order_max:
                    self.order_max = l

        self.radial_coeffs = radial_coeffs
        self.orbital_spec, self.radial_counts = self.combine_orbitals()
        self.init_radial_coeffs()

    def combine_orbitals(self):
        orbital_spec = [None] * len(self.orbitals)
        radial_counts = [None] * len(self.orbitals)
        for i in range(len(self.orbitals)):
            orbital_L_count = [0] * (self.order_max + 2)
            orbital_spec[i] = []
            radial_counts[i] = [[]] * (self.order_max + 2)
            # print('density L count len', len(density_L_count))
            z = self.orbitals[i][0][0]
            for j in range(len(self.orbitals[i])):
                orb = self.orbitals[i][j]
                L = orb[2]
                orbital_L_count[L] += 1
                radial_counts[i][L].append(orb[1])
            for L, c in enumerate(orbital_L_count):
                if c == 0:
                    continue
                orbital_spec[i].append((z, c, L))

        return orbital_spec, radial_counts

    def init_radial_coeffs(self):
        init_width = [None] * len(self.orbitals)
        init_scale = [None] * len(self.orbitals)
        for i in range(len(self.orbitals)):
            z = self.orbitals[i][0][0]
            init_width[i] = {}
            init_scale[i] = {}
            for j in range(len(self.orbitals[i])):
                orb = self.orbitals[i][j]
                L = orb[2]
                key = (z, L)
                width_coeff = torch.zeros((1, 1, max(self.radial_counts[i][L]), 1))
                scale_coeff = torch.zeros((1, 1, max(self.radial_counts[i][L]), 1))
                if self.radial_coeffs is not None:
                    n_coeff = len(self.radial_coeffs[i][j][0])
                    width_coeff[..., :n_coeff, 0] += torch.Tensor(self.radial_coeffs[i][j][0])
                    scale_coeff[..., :n_coeff, 0] += torch.Tensor(self.radial_coeffs[i][j][1])
                if key in init_width[i].keys():
                    init_width[i][key] = torch.cat((init_width[i][key], width_coeff), dim=-1)
                    init_scale[i][key] = torch.cat((init_scale[i][key], scale_coeff), dim=-1)
                else:
                    init_width[i][key] = width_coeff
                    init_scale[i][key] = scale_coeff

            for key in init_width[i].keys():
                self.register_buffer('init_width_{}_{}_{}'.format(i, key[0], key[1]), init_width[i][key])
                self.register_buffer('init_scale_{}_{}_{}'.format(i, key[0], key[1]), init_scale[i][key])

        return init_width, init_scale

    def init_width(self, i, key):
        return getattr(self, 'init_width_{}_{}_{}'.format(i, key[0], key[1]))

    def init_scale(self, i, key):
        return getattr(self, 'init_scale_{}_{}_{}'.format(i, key[0], key[1]))

    def forward(self, coords, atom_R, sph_coeffs, rad_width, rad_scale, eval_atoms=None, eval_L=None):
        # print('orbitals', self.orbitals)
        # print('coords', coords.shape)
        # print('atom_R shape', atom_R.shape)
        # print('rad_width keys', rad_width[0].keys())
        # print('rad_scale keys', rad_scale[0].keys())
        if eval_atoms is None:
            eval_atoms = list(range(len(self.orbitals)))
        if eval_L is None:
            eval_L = list(range(self.order_max + 1))
        result = {'density': 0}
        # print('eval atoms', eval_atoms)
        # print('eval L', eval_L)
        # print('order max', self.order_max)
        coeffs_sum = 0
        for i in eval_atoms:
            z = self.orbital_spec[i][0][0]
            coeffs_sum += torch.sum(sph_coeffs[i][(z, 0)])
        print("coeffs sum", coeffs_sum)
        print("n electrons", self.n_electrons)
        scale_factor = self.n_electrons/coeffs_sum
        for i in eval_atoms:
            z = self.orbital_spec[i][0][0]
            # print('atom num', z)
            # print('orbitals i', self.orbital_spec[i])
            d, u = calculate_distances_and_directions(coords, center=atom_R[:, [i]])
            s = spherical_harmonics(self.order_max, u)
            # print('atom[i]', i)
            # print('dists', d)
            # print('min dist', torch.min(d))
            for l in range(len(s)):
                zeros = torch.zeros_like(s[l])
                s[l] = torch.where(torch.isnan(s[l]), zeros, s[l])  # making sure there are no nans to avoid NaNs

            for orb in self.orbital_spec[i]:
                # print('orbital', orb)
                L = orb[2]
                if L not in eval_L:
                    continue
                key = (z, L)
                width = rad_width[i][key] + self.init_width(i, key)
                scale = rad_scale[i][key] + self.init_scale(i, key)
                sph_coeff = sph_coeffs[i][key]
                if L == 0:
                    sph_coeff *= scale_factor
                #print('num electrons', self.n_electrons)
                sph = s[L].unsqueeze(-1) * sph_coeff
                #print('sph_coeffs', sph_coeffs[i][key])
                #print('sph_coeffs norm', sph_coeffs[i][key]/sph_coeffs[i][key].norm(dim=-2))
                #print('sum sph coeffs', torch.sum(sph_coeff))
                #sph = s[L].unsqueeze(-1) * torch.ones_like(sph_coeffs[i][key])
                # print('sph nan', torch.sum(torch.isnan(sph)))
                #print('sph prod shape', sph.shape)
                rbf = gaussian_rbf(d.unsqueeze(-1), width, scale, True)
                # print('rbf nan', torch.sum(torch.isnan(rbf)))
                #print('rbf shape', rbf.shape)
                result['density'] += torch.sum(sph * rbf , dim=(-2, -1))
        if self.constraint_type == 'sq':
            result['density'] = result['density']**2
        if self.constraint_type == 'abs':
            result['density'] = torch.sqrt(result['density']**2)

        return result


def gaussian_rbf(r, width, scale, normalize=False):
    # print('scale shape', scale.shape)
    # print('scale shape', scale.shape)
    # print('width shape', width.shape)
    #print('r shape', r.shape)
    #print('norm factor', width**(3/2)/np.pi**(3/2))
    #print('norm factor shape', torch.sqrt(width / np.pi).shape)
    #print('analytic integral', (np.pi**(3/2))/(8*width**(3/2)))
    #print('width', width)
    #print('scale', scale)
    rbf = 8*(width**(3/2))/(np.pi**(3/2) * 53.9866) * torch.exp(-width * (r)**2)
    #rbf = scale * torch.exp(-width * (r)**2)
    #print('rbf shape', rbf.shape)
    #print('rbf', rbf)
    #print('sum_rbf', torch.sum(rbf, dim=1))
    return torch.sum(rbf, dim=-2, keepdim=True)


def calculate_distances_and_directions(r, center=None):
    if center is None:
        center = 0
    # print('r', r.type())
    # print('center', center.type())

    r = r - center
    d = torch.norm(r, dim=-1, keepdim=True)
    u = r / d
    return d, u

In [76]:
exp_model = TestSphericalHarmonicsExpansion(dataset.orbitals, radial_coeffs=dataset.radial_coeffs, constraint_type=args.expansion_constraint)

constraint type None


In [77]:
sample = 5
ds_sample = valid_cube_dataset[sample]
pos = ds_sample['positions']
print('positions', pos)
coords = ds_sample['coords']
weights = ds_sample['coord_weights']
print('coords shape', coords.shape)
print('weights shape', weights.shape)
dens_true = ds_sample['density']
print('density.shape', dens_true.shape)
print('density_integral', torch.sum(dens_true * weights))
print('dens_weights shape', (dens_true * weights).shape)
dens_true_integral = torch.sum(dens_true*weights)
coeffs = equiv_model(R=pos)
dens_pred = exp_model(coords, pos,
                            coeffs['spherical_coeffs'],
                            coeffs['radial_width'],
                            coeffs['radial_scale'])
dens_pred = dens_pred['density']
print('dens_pred', dens_pred)
print('density pred.shape', dens_pred.shape)
print('weights unsqueeze shape', weights.unsqueeze(0).unsqueeze(2).shape)
dens_pred_integral = torch.sum(dens_pred * weights)
#dens_pred_integral = torch.sum(dens_pred * weights.unsqueeze(0).unsqueeze(2), dim=[0, 1])
#dens_diff_integral = torch.sum(torch.abs(dens_pred - dens_true) * weights)
print('density pred integral', dens_pred_integral)
#print('density_difference integral', dens_diff_integral)
#print('density integral ratio', dens_diff_integral/dens_true_integral)
#dens_pred = exp_model(coords, pos,
#                            coeffs['spherical_coeffs'],
#                            coeffs['radial_width'],
#                          coeffs['radial_scale'], eval_atoms=[1], eval_L=[0])
#dens_pred = dens_pred['density']
#dens_pred_integral = torch.sum(dens_pred * weights)
#dens_diff_integral = torch.sum(torch.abs(dens_pred - dens_true) * weights)
#print('density pred integral', dens_pred_integral)
#print('density_difference integral', dens_diff_integral)
#print('density integral ratio', dens_diff_integral/dens_true_integral)

positions tensor([[[ 0.0000,  0.0000,  0.0000],
         [-0.9271,  0.1267, -0.5001],
         [ 0.0622, -0.6538, -0.4586]]])
coords shape torch.Size([1, 125000, 3])
weights shape torch.Size([125000])
density.shape torch.Size([1, 125000])
density_integral tensor(10.0396)
dens_weights shape torch.Size([1, 125000])
coeffs sum tensor(15.2611, grad_fn=<AddBackward0>)
n electrons 10
dens_pred tensor([[1.9573e-05, 2.1646e-05, 2.3853e-05,  ..., 7.7439e-07, 6.8027e-07,
         5.9490e-07]], grad_fn=<AddBackward0>)
density pred.shape torch.Size([1, 125000])
weights unsqueeze shape torch.Size([1, 125000, 1])
density pred integral tensor(77.4712, grad_fn=<SumBackward0>)


In [88]:
ds_sample = dataset[sample]
pos = ds_sample['positions']
print('positions', pos)
coords = ds_sample['coords']
print('coords shape', coords.shape)
weights = ds_sample['coord_weights']
dens_true = ds_sample['density']
print('density.shape', dens_true.shape)
print('density_integral', torch.sum(dens_true * weights))
print('dens_weights shape', (dens_true).shape)
dens_true_integral = torch.sum(dens_true * weights)
coeffs = equiv_model(R=pos)
dens_pred = exp_model(coords, pos,
                            coeffs['spherical_coeffs'],
                            coeffs['radial_width'],
                            coeffs['radial_scale'])
dens_pred = dens_pred['density']
#dens_pred_integral = torch.sum(dens_pred * weights.unsqueeze(0).unsqueeze(2), dim=[0, 1])
dens_pred_integral = torch.sum(dens_pred * weights)
print('density pred integral', dens_pred_integral)

positions tensor([[[ 0.0000,  0.0000,  0.0000],
         [-0.9271,  0.1267, -0.5001],
         [ 0.0622, -0.6538, -0.4586]]])
coords shape torch.Size([1, 91054, 3])
density.shape torch.Size([1, 91054])
density_integral tensor(9.8175)
dens_weights shape torch.Size([1, 91054])
coeffs sum tensor(15.2611, grad_fn=<AddBackward0>)
n electrons 10
density pred integral tensor(6.7348, grad_fn=<SumBackward0>)
